# Import library

In [1]:
import pandas as pd
from sqlalchemy import create_engine, URL
from dotenv import load_dotenv
from pathlib import Path
import os

# Membaca konfigurasi .env

In [2]:
load_dotenv(Path.cwd().parent / ".env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT"))
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

# Membuat koneksi database

In [3]:
connection_url = URL.create(
    drivername="mysql+mysqlconnector",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(connection_url)

# Menguji koneksi

In [4]:
with engine.connect() as connection:
    print("Berhasil terhubung ke MySQL!")

Berhasil terhubung ke MySQL!


# Menentukan tabel staging

In [5]:
staging_tables = [
    "stg_customers",
    "stg_products",
    "stg_orders",
    "stg_order_items",
    "stg_payments",
    "stg_shipments",
    "stg_returns",
    "stg_customer_reviews",
    "stg_marketing_campaigns"
]

# Melakukan Extract

In [6]:
data = {}

for table in staging_tables:
    query = f"SELECT * FROM `{table}`"

    data[table] = pd.read_sql_query(query, engine)

    print(f"{table}: {data[table].shape[0]} baris, "
          f"{data[table].shape[1]} kolom")

stg_customers: 25000 baris, 18 kolom
stg_products: 550 baris, 14 kolom
stg_orders: 100000 baris, 16 kolom
stg_order_items: 254331 baris, 9 kolom
stg_payments: 100000 baris, 8 kolom
stg_shipments: 89681 baris, 11 kolom
stg_returns: 12075 baris, 8 kolom
stg_customer_reviews: 77530 baris, 8 kolom
stg_marketing_campaigns: 151 baris, 14 kolom


# Membuat folder penyimpanan

In [7]:
from pathlib import Path

output_dir = Path.cwd().parent / "data" / "extracted"

output_dir.mkdir(parents=True, exist_ok=True)

print(f"Folder penyimpanan: {output_dir}")

Folder penyimpanan: c:\Projects\dwh_ecommerce\data\extracted


# Menyimpan seluruh DataFrame

In [8]:
for table_name, df in data.items():
    file_path = output_dir / f"{table_name}.parquet"

    df.to_parquet(file_path, index=False)

    print(f"Berhasil menyimpan: {table_name}.parquet")

Berhasil menyimpan: stg_customers.parquet
Berhasil menyimpan: stg_products.parquet
Berhasil menyimpan: stg_orders.parquet
Berhasil menyimpan: stg_order_items.parquet
Berhasil menyimpan: stg_payments.parquet
Berhasil menyimpan: stg_shipments.parquet
Berhasil menyimpan: stg_returns.parquet
Berhasil menyimpan: stg_customer_reviews.parquet
Berhasil menyimpan: stg_marketing_campaigns.parquet


# Memeriksa file yang tersimpan

In [11]:
for file in output_dir.glob("*.parquet"):
    print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

stg_customers.parquet - 764.35 KB
stg_customer_reviews.parquet - 1432.89 KB
stg_marketing_campaigns.parquet - 18.79 KB
stg_orders.parquet - 3636.3 KB
stg_order_items.parquet - 4588.3 KB
stg_payments.parquet - 2341.97 KB
stg_products.parquet - 39.78 KB
stg_returns.parquet - 298.13 KB
stg_shipments.parquet - 1726.14 KB


# Memeriksa hasil Extract

In [9]:
data["stg_customers"].head()

,customer_id,customer_signup_date,gender,age,age_group,state,city,pincode_prefix,customer_segment,preferred_device,preferred_payment_method,acquisition_channel,total_orders,total_spend,average_order_value,last_order_date,customer_status,loyalty_tier
0,CUST100000,2023-01-09,Male,44.0,35-44,Madhya Pradesh,Indore,607,Regular,Mobile App,EMI,Affiliate,3,36884.57,12294.86,2023-05-26,Churned,Bronze
1,CUST100001,2023-04-10,Male,20.0,18-24,West Bengal,Kolkata,245,Regular,Mobile App,Cash on Delivery,Organic Search,7,371585.73,53083.68,2024-05-23,Churned,Gold
2,CUST100002,2025-05-09,Female,35.0,35-44,Gujarat,Ahmedabad,160,Premium,Mobile Web,UPI,SMS Marketing,10,114901.74,11490.17,2025-07-14,At Risk,Silver
3,CUST100003,2024-12-24,Female,34.0,25-34,Maharashtra,Pune,175,New,Mobile App,UPI,Influencer Marketing,1,3788.82,3788.82,2025-12-19,Active,Bronze
4,CUST100004,2023-08-15,Male,38.0,35-44,Andhra Pradesh,Visakhapatnam,446,Budget,Desktop,Cash on Delivery,Referral,2,61295.90,30647.95,2023-11-01,Churned,Silver


In [10]:
data["stg_products"].head()

,product_id,product_name,category,subcategory,brand,price,cost_price,discount_range,rating_average,rating_count,stock_quantity,product_launch_date,product_type,return_rate_baseline
0,PROD10000,Sen Basic Staple,Grocery,Staples,Sen,220.0,173.88,0-15%,4.4,32,1422,2025-12-12,Premium,0.0118
1,PROD10001,Gupta-Chandra Essential Men's Bottomwear,Fashion,Men's Bottomwear,Gupta-Chandra,4060.0,2375.50,10-25%,5.0,13,1595,2023-09-01,Budget,0.1678
2,PROD10002,"Murty, Advanced Women's Casual",Footwear,Women's Casual,"Murty,",2800.0,1740.98,25-40%,4.2,139,732,2024-12-24,Premium,0.2193
3,PROD10003,Madan-Kanda Signature Bedding,Home & Kitchen,Bedding,Madan-Kanda,5680.0,4330.67,15-30%,4.1,31,1181,2024-03-28,Premium,0.0689
4,PROD10004,"Sami, Premium Home Decor",Home & Kitchen,Home Decor,"Sami,",370.0,250.98,25-40%,4.5,279,2597,2024-02-03,Standard,0.0664
